In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

def cosine_angular_loss(y_true, y_pred):
    y_true = tf.math.l2_normalize(y_true, axis=-1)
    y_pred = tf.math.l2_normalize(y_pred, axis=-1)
    cos = tf.reduce_sum(y_true * y_pred, axis=-1)
    cos = tf.clip_by_value(cos, -1.0, 1.0)
    return tf.reduce_mean(1.0 - cos)

huber = keras.losses.Huber(delta=1.0)

def build_multihead_model(img_shape=(224, 224, 3)):
    inp = keras.Input(shape=img_shape, name="image")

    backbone = keras.applications.EfficientNetV2B0(
        include_top=False,
        weights="imagenet",
        input_tensor=inp,
        pooling="avg",
    )

    x = backbone.output
    x = layers.Dense(512, activation="relu", name="head_dense_0")(x)
    x = layers.Dropout(0.3, name="head_dropout_0")(x)
    x = layers.Dense(256, activation="relu", name="head_dense_1")(x)
    x = layers.Dropout(0.2, name="head_dropout_1")(x)

    log_energy_out = layers.Dense(1, activation="linear", name="log_energy")(x)
    color_out = layers.Dense(3, activation="sigmoid", name="color")(x)

    to_light_raw = layers.Dense(3, activation="linear", name="to_light_raw")(x)
    to_light_out = layers.Lambda(
        lambda t: tf.math.l2_normalize(t, axis=-1),
        name="to_light_dir",
        output_shape=(3,)
    )(to_light_raw)

    log_dist_out = layers.Dense(1, activation="linear", name="log_dist")(x)

    light_dir_raw = layers.Dense(3, activation="linear", name="light_dir_raw")(x)
    light_dir_out = layers.Lambda(
        lambda t: tf.math.l2_normalize(t, axis=-1),
        name="light_dir",
        output_shape=(3,)
    )(light_dir_raw)

    cone_raw = layers.Dense(1, activation="linear", name="cone_raw")(x)
    spot_cone_out = layers.Lambda(
        lambda t: 90.0 * tf.sigmoid(t),
        name="spot_cone_deg",
        output_shape=(1,)
    )(cone_raw)

    blend_raw = layers.Dense(1, activation="linear", name="blend_raw")(x)
    spot_blend_out = layers.Lambda(
        lambda t: tf.sigmoid(t),
        name="spot_blend",
        output_shape=(1,)
    )(blend_raw)

    model = keras.Model(
        inputs=inp,
        outputs={
            "log_energy": log_energy_out,
            "color": color_out,
            "to_light_dir": to_light_out,
            "log_dist": log_dist_out,
            "light_dir": light_dir_out,
            "spot_cone_deg": spot_cone_out,
            "spot_blend": spot_blend_out,
        },
        name="lighting_predictor_multihead",
    )

    return model

model = build_multihead_model((224, 224, 3))

_ = model(tf.zeros((1, 224, 224, 3), dtype=tf.float32), training=False)

model.load_weights("lighting_model_spot_only.weights.h5")
print("Loaded weights OK.")

Loaded weights OK.


In [4]:
def load_image(path, size=(224, 224)):
    img = Image.open(path).convert("RGB")
    img = img.resize(size, Image.Resampling.LANCZOS)
    arr = np.asarray(img).astype(np.float32) / 255.0
    return arr[None, ...]

x = load_image("cylinder.png")

In [ ]:
pred = model.predict(x)

to_dir = pred["to_light_dir"][0]
dist = np.exp(pred["log_dist"][0, 0])
pos_cam = to_dir * dist

light_dir_cam = pred["light_dir"][0]
energy = float(np.exp(pred["log_energy"][0, 0]))
color = pred["color"][0]
cone = float(pred["spot_cone_deg"][0, 0])
blend = float(pred["spot_blend"][0, 0])

print("Camera-space prediction:")
print("pos:", pos_cam)
print("dir:", light_dir_cam)


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
Camera-space prediction:
pos: [-0.38498938  2.0401905   1.3293452 ]
dir: [-0.05858132 -0.620682    0.7818709 ]


In [ ]:
import json
import numpy as np

JSON_PATH = "test_camera_lights.json"

with open(JSON_PATH, "r", encoding="utf-8") as f:
    scene_data = json.load(f)

cam = scene_data["camera"]

cam_pos_world = np.array(cam["location_world"], dtype=np.float32)

R_cam = np.stack(
    [
        np.array(cam["right_world"], dtype=np.float32),
        np.array(cam["up_world"], dtype=np.float32),
        np.array(cam["forward_world"], dtype=np.float32),
    ],
    axis=1,
).astype(np.float32)

pos_world = cam_pos_world + (R_cam @ pos_cam)

dir_world = (R_cam @ light_dir_cam)
dir_world = dir_world / (np.linalg.norm(dir_world) + 1e-8)

print("\n=== Blender-ready (WORLD SPACE) ===")
print("light_pos_world =", pos_world.tolist())
print("light_dir_world =", dir_world.tolist())
print("energy =", energy)
print("color =", [float(color[0]), float(color[1]), float(color[2])])
print("cone_deg =", cone)
print("blend =", blend)

print("pos_cam z:", pos_cam[2])
print("||light_dir_cam||:", np.linalg.norm(light_dir_cam))

pos_cam_flipped = pos_cam.copy()
pos_cam_flipped[2] *= -1.0

light_dir_cam_flipped = light_dir_cam.copy()
light_dir_cam_flipped[2] *= -1.0

pos_world_flip = cam_pos_world + (R_cam @ pos_cam_flipped)
dir_world_flip = (R_cam @ light_dir_cam_flipped)
dir_world_flip /= (np.linalg.norm(dir_world_flip) + 1e-8)

print("FLIPPED VERSION:")
print("light_pos_world =", pos_world_flip.tolist())
print("light_dir_world =", dir_world_flip.tolist())



=== Blender-ready (WORLD SPACE) ===
light_pos_world = [5.567624092102051, -5.7663774490356445, 6.836996555328369]
light_dir_world = [-0.34850722551345825, 0.2480044662952423, -0.9039007425308228]
energy = 65.22138214111328
color = [0.9725773334503174, 0.9781957864761353, 0.9809327721595764]
cone_deg = 58.71532440185547
blend = 0.1668030172586441
pos_cam z: 1.3293452
||light_dir_cam||: 1.0
FLIPPED VERSION:
light_pos_world = [7.299915790557861, -7.399266242980957, 8.020835876464844]
light_dir_world = [0.6703616380691528, -0.712399423122406, -0.20761124789714813]
